In [ ]:
!pip install transformers
!pip install "transformers[torch]"

In [ ]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

: 

In [ ]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")


: 

In [ ]:
train_data.sample(10)

: 

In [1]:
# Random sampling
train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
val_data = val_data.sample(n=500, random_state=42).reset_index(drop=True)

NameError: name 'train_data' is not defined

In [ ]:
train_data.shape

## Data pre- processing

In [1]:
import re

def clean_data(text):
    text = re.sub(r"\r\n", " ", text) #remove lines
    text = re.sub(r"\s+", " ",text) # remove spaces
    text = re.sub(r"<.*?>", " ",text) # remove HTML tags <p> <h>
    text = text.strip().lower()
    return text
    

In [2]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

NameError: name 'train_data' is not defined

In [ ]:
train_data["dialogue"][0]

### Tokenize

In [3]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

NameError: name 'T5Tokenizer' is not defined

In [ ]:
# raw data =. tokenized inputs for fine-tuning

def tokenize(data):
    inputs = tokenizer(data["dialogue"], padding="max_length", max_length = 512, truncation=True)
    targets = tokenizer(data["summary"], padding="max_length", max_length = 150, truncation=True)

    inputs["labels"] = targets["input_ids"] #token ids  => add to input as labels
    return inputs

In [ ]:
train_dataset = train_data.apply(tokenize, axis=1).tolist() # convert to list bcz list is compatible with huggingface
val_dataset = val_data.apply(tokenize, axis=1).tolist()

In [ ]:
type(train_dataset[0]["input_ids"])
type(train_dataset[0]["input_ids"])

### Working with our model

In [ ]:
# NLP => generation task
# T5ForConditionalGeneration for conditional genration task used in this project to get summary from given texts

model = T5ForConditionalGeneration.from_pretrained("t5-small")

In [4]:
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")      # Apple Silicon GPU
elif torch.cuda.is_available():
    device = torch.device("cuda")     # NVIDIA GPU
else:
    device = torch.device("cpu")      # CPU

print("Device:", device)

model.to(device)

ModuleNotFoundError: No module named 'torch'

In [5]:
# Training Arguments

training_args = TrainingArguments(
    output_dir = "./results",

    num_train_epochs = 6,
    weight_decay = 0.01,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy="epoch",
    save_strategy="epoch",

    warmup_steps = 500,
    # 0 => lr default
)

NameError: name 'TrainingArguments' is not defined

### Trainer

In [6]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset
)

NameError: name 'Trainer' is not defined

### Save the model

In [7]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

NameError: name 'model' is not defined

### Test the core logic for summarization

In [8]:
def summarize_dialogue(dialogue):
    dialogue = clean_data(dialogue) # clean

    # tokenize
    inputs = tokenizer(
        dialogue,
        padding = "max_length",
        max_length = 512,
        truncation = True,
        return_tensors = "pt"
    )

    
    inputs = {key: value.to(device) for key, value in inputs.items()} # our model and data should on same device

    # generate the summary => token ids
    targets = model.generate(
        input_ids  = inputs["input_ids"], #token ids
        attention_mask = inputs["attention_mask"], # show amount of paddings
        max_length = 150,
        num_beams = 4, #transformers gives 4 different summaries and will give the best among them
        early_stopping = True, # 
        
    )

    #decode our output
    # token ids  converts to text summary => decoding 
    summary = tokenizer.decode(targets[0], skip_special_tokens = True) # EOS , SEP
    return summary